Install Packages

In [1]:
!python3 -m pip install pandas numpy yfinance ipywidgets

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


Imports & Setup

In [2]:
import sys
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
import ipywidgets as widgets
from IPython.display import display
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore", category=FutureWarning)

Configuration & Tickers

In [3]:
# CONFIGURATION & TICKERS
TICKERS = [
    "NVDA","GOOG","AAPL","MSFT","AMZN","TSM","META","AVGO","TSLA","LLY","WMT","JPM","V","XOM","ASML","JNJ","ORCL","MA","MU","AMD","PLTR","BABA","ABBV","HD","BAC",
    "NFLX","PG","CVX","UNH","KO","GE","CSCO","CAT","NVS","TM","SAP","HSBC","AZN","MS","NVO","LRCX","GS","WFC","IBM","PM","MRK","RTX","AMAT","AXP","TMO","CCZ",
    "RY","MCD","CRM","SHEL","INTC","LIN","MUFG","TMUS","C","KLAC","PEP","DIS","APH","BA","ISRG","ABT","AMGN","ACN","SAN","GEV","APP","TXN","NEE","SHOP",
    "BHP","UBER","GILD","DHR","VZ","T","TJX","QCOM","HDB","BKNG","TD","SPGI","TBB","INTU","LOW","SCCO","TTE","PDD","UL","ADI","PFE","BBVA","NOW","HON",
    "COF","BSX","DE","SONY","NEM","UNP","SYK","SMFG","LMT","MDT","ETN","BTI","PANW","WELL","ADBE","ARM","COP","VRTX","SATA","PH","CRWD","PLD","SNY","RIO",
    "MELI","BMY","BX","AEM","SBUX","HCA","CMCSA","MFG","KKR","BN","CVS","SPOT","IBN","ENB","CVNA","CEG","CME","GSK","MCK","ICE","GD","SO","SNPS","NKE",
    "HOOD","BP","NOC","ITUB","MCO","BNS","DUK","WM","PBR","BCS","UPS","DASH","MRSH","FCX","CDNS","USB","B","NU","HWM","PNC","SHW","CM","MMM","ORLY",
    "MAR","LYG","AMT","NTES","EMR","BK","BAM","WDC","CRH","NGG","ABNB","GLW","TDG","ECL","REGN","MNST","RCL","EQIX","CMI","WMB","CTAS","INFY","DELL",
    "STX","DB","APO","EQNR","MDLZ","ITW","CI","GM","ELVR","SLB","AON","SNOW","RELX","NWG","FDX","EPD","JCI","PWR","VRT","SNDK","MRVL","COR","HLT","CL",
    "RSG","NET","MSI","VALE","TEL","LHX","E","KMI","CP","AJG","NSC","TFC","PCAR","AMX","AEP","AZO","TRV","MFC","ET","CNI","SU","BRKRP","ROST","RKT",
    "MDLN","APD","TRP","NXPI","URI","ADSK","COIN","AFL","NDAQ","PSX","SRE","DLR","BKR","VLO","IDXX","MPLX","AU","SOMN","ZTS","TRI","VST","F","BIDU",
    "PYPL","CCJ","CMG","MPC","RBLX","D","FERG","MPWR","ALL","EA","AME","MET","CBRE","ARGX","FNV","IMO","BSBR","WDAY","FAST","DEO","GWW","GFI","CRWV",
    "CTVA","FER","OKE","AXON","DDOG","ALNY","TGT","ROK","HEI","MSTR","KGC","HLN","TTWO","EXC","XEL","MSCI","DAL","RKLB","ROP","WCN","ASX","FANG",
    "ABEV","HMC","DHI","OXY","JD","EBAY","YUM","PRU","ETR","EL","BBD","TCOM","CTSH","MT","LVS","PUK","RDDT","CCEP","HBANL","PCG","IQV","MCHP","KR",
    "XYZ","ALC","GRMN","FIX","AIG","WAB","PEG","ASTS","MLM","CCL","NOK","CUK","A","HSY","PAYX","CCI","ED"
]

# Indicator Periods & Settings
RSI_PERIOD_WEEKLY = [12, 15, 17]
RSI_PERIOD_3D = [13, 16, 19]
RS_PERIOD_WEEKLY = [12, 15, 21]
ROC_PERIOD = [3, 9]
ATH_PERIOD = [21, 35, 55]

# Weights & Multipliers
MULTIPLIERS = [0.35, 0.45, 0.20]
RS_MULTIPLIERS = [0.33, 0.42, 0.25]
ATH_MULTIPLIERS = [0.95, 0.95]
RANK_WEIGHTS = [0.36, 0.28, 0.24, 0.12]
W_HIGH = 0.55
W_LOW = 0.45

# Thresholds & Filters
RSI_FILTER = [-13, 74, 60, 87]
ROC_FILTER = [90, 60]
RANK_FILTERS = [90, 90, 90, 90]
WEEKLY_GAIN_LIMIT = 0.27
SHIFTS = [30, 22, 14]

# Smoothing Windows
EMA_WINDOW_SIZE = 3
RMA_WINDOW_SIZE = 2

Date Input

In [4]:
# USER INPUT
print("PICK AN END DATE (YYYY-MM-DD) [must be a Saturday or Sunday, e.g., 2024-01-27]:")
print("Use the date [05-07 of month Feb,May,Aug,Nov] & [21-22 of month Mar,Jun,Sep,Dec]:")

end_date_input = widgets.DatePicker(
  description='START DATE',
  disabled=False
)
display(end_date_input)

PICK AN END DATE (YYYY-MM-DD) [must be a Saturday or Sunday, e.g., 2024-01-27]:
Use the date [05-07 of month Feb,May,Aug,Nov] & [21-22 of month Mar,Jun,Sep,Dec]:


DatePicker(value=None, description='START DATE', step=1)

Helper Functions

In [5]:
# HELPER FUNCTIONS
def fetch_adjusted_close(ticker: str, start_date: str, end_date: str) -> pd.DataFrame:
    """Fetches adjusted close prices for a given ticker."""
    df = yf.download(ticker, start=start_date, end=end_date, auto_adjust=True, progress=False)
    if df.empty:
        raise ValueError(f"No data for {ticker}")
    df.columns = df.columns.get_level_values(0)
    return df[['Close']]

def build_price_series(stock_close: pd.DataFrame, benchmark_close: pd.DataFrame = None, mode: str = "RAW") -> pd.DataFrame:
    """Builds either a raw price series or a relative series against a benchmark."""
    if mode == "RAW":
        return stock_close.rename(columns={"Close": "Price"})
    if mode == "RELATIVE":
        if benchmark_close is None:
            raise ValueError("Benchmark required for RELATIVE mode")
        merged = stock_close.merge(benchmark_close, left_index=True, right_index=True, how="inner")
        merged["Price"] = merged["Close_x"] / merged["Close_y"]
        return merged[["Price"]]
    raise ValueError("Invalid price mode")

def resample_weekly(price_df: pd.DataFrame) -> pd.DataFrame:
    """Resamples daily data to weekly data (using a 7-trading-day stride)."""
    return (
        price_df
        .iloc[::-1].iloc[::7].iloc[::-1] # Note: Consider resample("W-FRI") for calendar weeks
        .rename(columns={"Price": "W_Closed"})
        .reset_index()
    )

def resample_3day(price_df: pd.DataFrame) -> pd.DataFrame:
    """Resamples daily data to 3-day data (using a 3-trading-day stride)."""
    return (
        price_df
        .iloc[::-1].iloc[::3].iloc[::-1] # Note: Consider resample("3B") for calendar 3-day windows
        .rename(columns={"Price": "3D_Closed"})
        .reset_index()
    )

def compute_rsi_series(series: pd.Series, period: int) -> pd.Series:
    """Computes the Relative Strength Index (RSI)."""
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

def compute_weighted_rsi(price_df: pd.DataFrame, price_col: str, rsi_periods: list[int], multipliers: list[float]) -> pd.Series:
    """Calculates a weighted average of RSI values over multiple periods."""
    if len(rsi_periods) != len(multipliers):
        raise ValueError("RSI periods and multipliers must have the same length")
    
    combined_rsi = pd.Series(0.0, index=price_df.index, dtype="float64")
    for period, weight in zip(rsi_periods, multipliers):
        rsi = compute_rsi_series(price_df[price_col], period)
        combined_rsi += weight * rsi
    return combined_rsi

def compute_price_distance(series: pd.Series, shifts: list[int], window: int, w_high: float, w_low: float, prefix: str) -> pd.DataFrame:
    """Computes the percentage distance from a custom rolling channel."""
    out = pd.DataFrame(index=series.index)
    for s in shifts:
        shifted = series.shift(s)
        hi = shifted.rolling(window, min_periods=1).max()
        lo = shifted.rolling(window, min_periods=1).min()
        denom = w_high * hi + w_low * lo
        out[f"{prefix}_{s}"] = series.div(denom).sub(1) * 100
    return out

def compute_Rate_of_Change(price: pd.Series, prefix: str = "ROC") -> pd.DataFrame:
    """Computes the Rate of Change (Momentum) over specific periods."""
    out = pd.DataFrame(index=price.index)
    for s in ROC_PERIOD:
        shifted = price.shift(s)
        out[f"{prefix}_{s}"] = price.div(shifted).sub(1) * 100
    return out

def compute_rma_ratios(rma_series: pd.Series, shifts: list[int]) -> pd.DataFrame:
    """Computes the ratio of the current RMA to shifted RMA values."""
    out = pd.DataFrame(index=rma_series.index)
    for s in shifts:
        out[f"RS_{s}"] = rma_series / rma_series.shift(s)
    return out

def latest(df):
    """Retrieves the latest available data row per ticker."""
    return df.sort_values("Date").groupby("Ticker").tail(1).set_index("Ticker")

Main Execution Engine

In [6]:
# MAIN EXECUTION
run_ok = False

try:
    if end_date_input.value is None:
        raise ValueError("END DATE is required and must be a weekend (Saturday or Sunday).")

    end_date = end_date_input.value

    if end_date.weekday() not in (5, 6):
        raise ValueError(f"END DATE {end_date} is not a weekend. Please provide a Saturday or Sunday.")
    
    start_date = (end_date - timedelta(days=600)).strftime('%Y-%m-%d')

    # Fetch Benchmark Data (^NDX - Nasdaq 100)
    print("Fetching benchmark data...")
    benchmark_close = fetch_adjusted_close("^NDX", start_date, end_date)
    benchmark_index = benchmark_close.index
    benchmark_price = build_price_series(benchmark_close, mode="RAW")
    
    benchmark_W = resample_weekly(benchmark_price)
    benchmark_W["RMA_W"] = benchmark_W["W_Closed"].rolling(window=RMA_WINDOW_SIZE).mean()
    benchmark_W["EMA_W"] = benchmark_W["W_Closed"].ewm(span=15, adjust=False).mean()
    benchmark_W = benchmark_W.join(compute_rma_ratios(benchmark_W["RMA_W"], RS_PERIOD_WEEKLY))
    run_ok = True

except Exception as e:
    print(f"Critical Error: Failed to fetch benchmark data: {e}")

if run_ok:
    if benchmark_W["RMA_W"].iloc[-1] <= benchmark_W["EMA_W"].iloc[-1]:
        print("Market is not investment friendly (Bear Regime). Exiting.")
    else:
        print("Market is favorable. Processing tickers...")
        factor_values = {}
        
        # Lists for storing historical data across all tickers
        raw_W_all, rel_W_all, final_RSI_W_all = [], [], []
        raw_3D_all, rel_3D_all, final_RSI_3D_all = [], [], []
        
        for ticker in TICKERS:
            try:
                stock_close = fetch_adjusted_close(ticker, start_date, end_date).reindex(benchmark_index).ffill()
                if stock_close.empty or len(stock_close) < 200:
                    print(f"Skipping {ticker}: Insufficient historical data.")
                    continue

                # Series Generation
                raw_price = build_price_series(stock_close, mode="RAW")
                rel_price = build_price_series(stock_close, benchmark_close, mode="RELATIVE")

                raw_W = resample_weekly(raw_price)
                raw_3D = resample_3day(raw_price)
                rel_W = resample_weekly(rel_price)
                rel_3D = resample_3day(rel_price)

                # RSI Calculations
                raw_W["RSI_COMBINED"] = compute_weighted_rsi(raw_W, "W_Closed", RSI_PERIOD_WEEKLY, MULTIPLIERS)
                raw_3D["RSI_COMBINED"] = compute_weighted_rsi(raw_3D, "3D_Closed", RSI_PERIOD_3D, MULTIPLIERS)
                rel_W["RSI_COMBINED"] = compute_weighted_rsi(rel_W, "W_Closed", RSI_PERIOD_WEEKLY, MULTIPLIERS)
                rel_3D["RSI_COMBINED"] = compute_weighted_rsi(rel_3D, "3D_Closed", RSI_PERIOD_3D, MULTIPLIERS)

                final_RSI_W = pd.DataFrame(index=raw_W.index)
                final_RSI_3D = pd.DataFrame(index=raw_3D.index)
                final_RSI_W["RSI_W"] = raw_W["RSI_COMBINED"] * W_HIGH + rel_W["RSI_COMBINED"] * W_LOW
                final_RSI_3D["RSI_3D"] = raw_3D["RSI_COMBINED"] * W_HIGH + rel_3D["RSI_COMBINED"] * W_LOW

                raw_W["EMA_RSI_W"] = raw_W["RSI_COMBINED"].ewm(span=EMA_WINDOW_SIZE, adjust=False).mean()
                final_RSI_W["EMA_RSI_W"] = final_RSI_W["RSI_W"].ewm(span=EMA_WINDOW_SIZE, adjust=False).mean()
                final_RSI_3D["EMA_RSI_3D"] = final_RSI_3D["RSI_3D"].ewm(span=EMA_WINDOW_SIZE, adjust=False).mean()
                final_RSI_W["S_EMA_RSI_W"] = final_RSI_W["RSI_W"].ewm(span=9, adjust=False).mean()

                raw_W["RSI_ATH_D"] = raw_W["RSI_COMBINED"] - raw_W["EMA_RSI_W"].shift(2).rolling(window=31, min_periods=1).max()
                final_RSI_W["RSI_ATH_D"] = final_RSI_W["RSI_W"] - final_RSI_W["EMA_RSI_W"].shift(2).rolling(window=31, min_periods=1).max()
                final_RSI_W["High"] = (final_RSI_W["RSI_ATH_D"] >= final_RSI_W["RSI_ATH_D"].rolling(ATH_PERIOD[0], min_periods=ATH_PERIOD[0]).max() * ATH_MULTIPLIERS[0]).astype(int)

                # Distance & Momentum
                rel_3D_dist = compute_price_distance(rel_3D["3D_Closed"], SHIFTS, 5, W_HIGH, W_LOW, "DIS")
                rel_3D["DIS_FINAL"] = sum(rel_3D_dist[f"DIS_{s}"] * w for s, w in zip(SHIFTS, MULTIPLIERS))

                raw_W = raw_W.join(compute_Rate_of_Change(raw_W["W_Closed"]))
                raw_W["W_DIS_FINAL"] = (
                    0.57 * (raw_W["W_Closed"] / raw_W["W_Closed"].ewm(span=11, adjust=False).mean() - 1) +
                    0.43 * (raw_W["W_Closed"] / raw_W["W_Closed"].ewm(span=19, adjust=False).mean() - 1)
                ) * 100

                # Relative Strength Processing
                raw_W["RMA_W"] = raw_W["W_Closed"].rolling(window=RMA_WINDOW_SIZE).mean()
                raw_W = raw_W.join(compute_rma_ratios(raw_W["RMA_W"], RS_PERIOD_WEEKLY))

                for rsp in RS_PERIOD_WEEKLY:
                    raw_W[f"Final_RS_{rsp}"] = raw_W[f"RS_{rsp}"] / benchmark_W[f"RS_{rsp}"]

                raw_W["PCT_CHANGE"] = raw_W["W_Closed"].pct_change()

                conditions = (
                    (raw_W["PCT_CHANGE"] < WEEKLY_GAIN_LIMIT) &
                    (raw_W["PCT_CHANGE"].shift(1) < WEEKLY_GAIN_LIMIT + 0.01) &
                    (raw_W["PCT_CHANGE"].shift(2) < WEEKLY_GAIN_LIMIT + 0.02) &
                    (raw_W["PCT_CHANGE"].shift(3) < WEEKLY_GAIN_LIMIT + 0.04) &
                    (raw_W["PCT_CHANGE"].shift(4) < WEEKLY_GAIN_LIMIT + 0.06)
                )
                raw_W["CHECK"] = conditions.astype(int)
                raw_W["High"] = (raw_W['W_Closed'] >= raw_W['W_Closed'].rolling(ATH_PERIOD[0], min_periods=ATH_PERIOD[0]).max() * ATH_MULTIPLIERS[0]).astype(int)

                rel_W["RMA_W"] = rel_W["W_Closed"].rolling(window=RMA_WINDOW_SIZE).mean()
                rel_W["High"] = (rel_W["RMA_W"] >= rel_W["RMA_W"].rolling(ATH_PERIOD[1], min_periods=ATH_PERIOD[1]).max() * ATH_MULTIPLIERS[1]).astype(int)

                raw_W["Total_RS"] = (raw_W[f"Final_RS_{RS_PERIOD_WEEKLY[0]}"] * RS_MULTIPLIERS[0] + 
                                     raw_W[f"Final_RS_{RS_PERIOD_WEEKLY[2]}"] * RS_MULTIPLIERS[1] + 
                                     raw_W[f"Final_RS_{RS_PERIOD_WEEKLY[2]}"] * RS_MULTIPLIERS[2])
                
                raw_W["RMA_RS"] = raw_W["Total_RS"].rolling(window=RMA_WINDOW_SIZE).mean()
                raw_W["EMA_RS"] = raw_W["Total_RS"].ewm(span=9, adjust=False).mean()

                # Collect Latest Features
                factor_values[ticker] = {
                    "Close": raw_W["W_Closed"].iloc[-1],
                    "RSI_3D": final_RSI_3D["EMA_RSI_3D"].iloc[-1],
                    "RSI_FW": final_RSI_W["EMA_RSI_W"].iloc[-1],
                    "RSI_W": raw_W["EMA_RSI_W"].iloc[-1],
                    "RSI_ATH": final_RSI_W["High"].iloc[-1],
                    "RS_SHORT": raw_W[f"Final_RS_{RS_PERIOD_WEEKLY[0]}"].iloc[-1],
                    "RS_MID": raw_W[f"Final_RS_{RS_PERIOD_WEEKLY[1]}"].iloc[-1],
                    "RS_LONG": raw_W[f"Final_RS_{RS_PERIOD_WEEKLY[2]}"].iloc[-1],
                    "RS_R": raw_W["RMA_RS"].iloc[-1],
                    "RS_E": raw_W["EMA_RS"].iloc[-1],
                    "ROC_LONG": raw_W[f"ROC_{ROC_PERIOD[1]}"].iloc[-1],
                    "ROC_DIFF":  raw_W[f"ROC_{ROC_PERIOD[1]}"].iloc[-1] - raw_W[f"ROC_{ROC_PERIOD[1]}"].shift(6).iloc[-1],
                    "DIS_RANK": raw_W["W_DIS_FINAL"].iloc[-1],
                    "DIS_DIFF": raw_W["W_DIS_FINAL"].iloc[-1] - raw_W["W_DIS_FINAL"].shift(5).iloc[-1]
                }

                # Add metadata for aggregation
                for df, lst in [(raw_W, raw_W_all), (rel_W, rel_W_all), (final_RSI_W, final_RSI_W_all), 
                                (raw_3D, raw_3D_all), (rel_3D, rel_3D_all), (final_RSI_3D, final_RSI_3D_all)]:
                    df_tmp = df.copy()
                    df_tmp["Ticker"] = ticker
                    df_tmp["Date"] = df_tmp.index
                    lst.append(df_tmp)

            except Exception as e:
                # Silently continue on errors to maintain screen continuity 
                continue
                
        # ==========================================
        # AGGREGATION & RANKING
        # ==========================================
        print("Scoring and Ranking...")
        W_raw = pd.concat(raw_W_all, ignore_index=True)
        W_rel = pd.concat(rel_W_all, ignore_index=True)
        W_final_RSI = pd.concat(final_RSI_W_all, ignore_index=True)
        D3_final_RSI = pd.concat(final_RSI_3D_all, ignore_index=True)

        latest_df = pd.DataFrame.from_dict(factor_values, orient="index")
        factor_cols = latest_df.columns.drop(["Close", "DIS_DIFF", "ROC_DIFF", "RSI_ATH"])

        # Ranking calculation
        latest_rank = latest_df[factor_cols].rank(method="min", ascending=False)
        
        latest_rank["Final_RSI_Rank"] = ((latest_rank["RSI_3D"] * 0.45 + latest_rank["RSI_FW"] * 0.55) * 0.53) + latest_rank["RSI_W"] * 0.47
        latest_rank["Final_RS_Rank"] = latest_rank["RS_SHORT"] * RS_MULTIPLIERS[0] + latest_rank["RS_MID"] * RS_MULTIPLIERS[1] + latest_rank["RS_LONG"] * RS_MULTIPLIERS[2]

        adj = pd.Series(0, index=latest_rank.index)

        # Rank Adjustments
        adj[latest_df["RSI_ATH"] == 1] -= 3
        latest_rank["Final_RSI_Rank"] = (latest_rank["Final_RSI_Rank"] + adj).clip(lower=1)

        adj[:] = 0
        adj[latest_df["ROC_DIFF"] <= -7] += 3
        adj[(latest_df["ROC_DIFF"] >= 10) & (latest_rank["ROC_LONG"] >= 10)] -= 3
        latest_rank["Final_ROC_Rank"] = (latest_rank["ROC_LONG"] + adj).clip(lower=1)

        adj[:] = 0
        adj[latest_df["DIS_DIFF"] <= -5] += 3
        adj[(latest_df["DIS_DIFF"] >= 8) & (latest_rank["DIS_RANK"] >= 10)] -= 3
        latest_rank["Final_DIS_Rank"] = (latest_rank["DIS_RANK"] + adj).clip(lower=1)

        latest_rank["Final_Rank"] = (latest_rank["Final_RSI_Rank"] * RANK_WEIGHTS[0] +
                                     latest_rank["Final_RS_Rank"] * RANK_WEIGHTS[1] +
                                     latest_rank["Final_ROC_Rank"] * RANK_WEIGHTS[2] +
                                     latest_rank["Final_DIS_Rank"] * RANK_WEIGHTS[3])

        final_output = pd.DataFrame({"Final_Rank": latest_rank["Final_Rank"]})
        idx = final_output.index

        latest_W = latest(W_raw)
        latest_W_R = latest(W_rel)
        latest_RSI_W = latest(W_final_RSI)
        latest_RSI_3D = latest(D3_final_RSI)

        # Apply Filters
        common_filter = (((latest_W["High"] == 1) | (latest_W_R["High"] == 1)) & (latest_W["W_Closed"] >= 5)).reindex(idx).fillna(False)

        latest_combined = latest_W.join(
            latest_RSI_W[["RSI_ATH_D", "EMA_RSI_W", "S_EMA_RSI_W"]], rsuffix="_weighted", how="inner"
        ).join(
            latest_RSI_3D[["EMA_RSI_3D"]], how="inner"
        )

        rsi_filter = (
            (latest_combined["RSI_ATH_D"] >= RSI_FILTER[0]) & 
            (latest_combined["RSI_ATH_D_weighted"] >= (RSI_FILTER[0] - 3)) &
            (((latest_combined["EMA_RSI_W"] >= RSI_FILTER[1])) | 
             ((latest_combined["EMA_RSI_W"] >= latest_combined["S_EMA_RSI_W"]) & (latest_combined["EMA_RSI_W"] <= RSI_FILTER[1]))) &
            ((latest_combined["EMA_RSI_W"] >= RSI_FILTER[2]) & (latest_combined["EMA_RSI_W"] <= RSI_FILTER[3]) & 
             (latest_combined["EMA_RSI_W_weighted"] >= RSI_FILTER[2]))
        ).reindex(idx).fillna(False)

        rs_filter = (latest_W["RMA_RS"] >= latest_W["EMA_RS"] * 0.95).reindex(idx).fillna(False)

        roc_filter = (
            (latest_W[f"ROC_{ROC_PERIOD[1]}"] <= ROC_FILTER[0]) &
            (latest_W["W_DIS_FINAL"] <= ROC_FILTER[1])
        ).reindex(idx).fillna(False)

        rank_filter = (
            (latest_rank["Final_RSI_Rank"] <= RANK_FILTERS[0]) &
            (latest_rank["Final_RS_Rank"] <= RANK_FILTERS[1]) &
            (latest_rank["Final_ROC_Rank"] <= RANK_FILTERS[2]) &
            (latest_rank["Final_DIS_Rank"] <= RANK_FILTERS[3])
        ).reindex(idx).fillna(False)

        # Output final selection
        final_filter = rank_filter & rsi_filter & common_filter & rs_filter & roc_filter
        final_Stocks = final_output[final_filter].sort_values("Final_Rank").head(20)

        print("\n--- TOP 20 STOCKS ---")
        print(final_Stocks)

        # Export to CSV
        final_Stocks.reset_index().rename(columns={"index": "Ticker"}).to_csv("Final_Stocks.csv", index=False)
        print("\nSuccessfully saved to final_Stocks.csv")

Fetching benchmark data...
Market is favorable. Processing tickers...
Scoring and Ranking...

--- TOP 20 STOCKS ---
      Final_Rank
B        6.12624
KGC      9.21074
LRCX     9.26230
AU      10.33714
SCCO    12.92860
AMAT    15.04786
MT      15.37306
NEM     15.52342
ASX     15.64388
VALE    17.53496
CMI     20.32772
KLAC    21.04014
RKLB    22.42834
ASML    22.78350
AEM     25.79404
RIO     26.75902
GOOG    26.84778
CCJ     28.97348
ASTS    31.10886
MFG     31.75450

Successfully saved to final_Stocks.csv
